
token 23
 http://127.0.0.1:9997/lab?token=f92d7bda0618c680b91069ba9333dae4686b5bac989f4756



In [ ]:
 !pip install ollama






In [ ]:
import numpy as np
import pandas as pd
from sklearn.neighbors import NearestNeighbors
from sklearn import preprocessing
import scipy
import sympy
from sympy import *
from sympy.solvers.ode.systems import dsolve_system
from sympy import init_printing
init_printing()
from sympy import symbols, Eq, Function, dsolve
from sympy import init_printing
init_printing()
import math
from scipy.integrate import odeint
from scipy.integrate import odeint
from scipy.integrate import solve_ivp  # or nothing
import os
import random
import torch
import json
import requests
import time
import hashlib
from typing import List, Dict, Any, Tuple, Literal, Union
from tqdm import tqdm
import ast
from requests.exceptions import ConnectionError
from http.client import RemoteDisconnected
#from x_llm import llm_query
import ollama
import weakref
from google.colab import drive

In [ ]:
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pwd

/content


In [ ]:
DEFAULT_MODEL_PARAMS = {'top_k':1,'temperature':0.1, 'top_p':0.0001}

# CACHE_PATH = os.path.join( 'tmp', "cache_llm_respostas_ANALISES.json" )
CACHE_PATH = "/content/drive/MyDrive/codigo_python/cache_llm_respostas_ANALISES.json"
# HOST_OLLAMA = "10.10.10.106:11434"
HOST_OLLAMA = 'local'
SAVE_CACHE_INTERVAL=1
SEED = 42
DEFAULT_TIME_SLEEP = 1

def fixar_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    # torch.manual_seed(seed)

    # if torch.cuda.is_available():
    #     # torch.cuda.manual_seed(seed)
    #     # torch.cuda.manual_seed_all(seed)  # para múltiplas GPUs
    #     torch.backends.cudnn.deterministic = True
    #     torch.backends.cudnn.benchmark = False



LLMModelsType = Union[
    str,
    Tuple[str, Literal['ollama', 'openrouter']]
]

class LLMInference:
    def __init__(self, api_key: str, models: List[LLMModelsType] = [], models_params: List[Dict[str, str]] = None,
                 system_prompt: str = None,
                 structured_output_instructions: Dict[str, Any] = None,
                 use_cache: bool = True,
                 cache_interval:int = 10,
                 cache_path: str = CACHE_PATH,
                 default_model_type:Literal['ollama', 'openrouter'] = 'ollama',
                 fix_seed:int = 42,
                 host_ollama: str = HOST_OLLAMA,
                 default_time_sleep: int = DEFAULT_TIME_SLEEP):
        """
        Initialize the batch inference class using OpenAI client for OpenRouter.

        Args:
            api_key (str): Your OpenRouter API key.
            models (List[str]): List of model names to query.
            system_prompt (str): Common system prompt for all queries.
            structured_output_instructions (Dict[str, Any]): Instructions to guide the output structure.
        """

        self.models = models
        if models_params is None:
            self.models_params = [DEFAULT_MODEL_PARAMS]*len(models)
        else:
            self.models_params = models_params
        self.system_prompt = system_prompt
        self.structured_output_instructions = structured_output_instructions
        self.api_key = api_key

        if fix_seed is not None:
            fixar_seed( fix_seed )

        self.seed = fix_seed
        self.default_model_type = default_model_type
        self.use_cache = use_cache
        self.cache_interval = cache_interval
        self.cache_path = cache_path
        self.host_ollama = host_ollama
        self.cache_counter = 0
        self.default_time_sleep = default_time_sleep

        if os.path.exists(self.cache_path):
            with open(self.cache_path) as f:
                self.cache = json.load(f)
                print('cache lido', flush=True)
        else:
            self.cache = {}

        weakref.finalize(self, self._save_cache)


    def _create_messages(self, user_prompt: str) -> List[Dict[str, str]]:
        """
        Create the message payload.

        Args:
            user_prompt (str): The user-specific prompt.

        Returns:
            List[Dict[str, str]]: Messages to send to the model.
        """
        messages = []
        if self.system_prompt:
            messages.append( {"role": "system", "content": self.system_prompt} )
        messages.append({"role": "user", "content": user_prompt})
        if self.structured_output_instructions:
            messages.append( {"role": "user", "content": f"Format your output according to: {self.structured_output_instructions}"} )
        return messages

    def _query_model_raw_openrouter(self, model: str, model_params: Dict[str,Any], user_prompt: str, max_retries:int=5) -> str:
        """
        Query a single model using OpenAI client.

        Args:
            model (str): Model name.
            user_prompt (str): User-specific prompt.

        Returns:
            str: Model's output content.
        """
        # completion = self.client.chat.completions.create(
        #     model=model,
        #     messages=self._create_messages(user_prompt),
        #     top_k=1,
        #     temperature=0.1
        # )
        # return completion.choices[0].message.content

        url = "https://openrouter.ai/api/v1/chat/completions"
        headers = {
        "Authorization": f"Bearer {self.api_key}",
        "Content-Type": "application/json"
        }
        payload = {
        "model": model,
        "messages": self._create_messages(user_prompt)
        }
        payload.update(model_params)


        retry_count = 0

        while True:
            try:
                response = requests.post(url, headers=headers, json=payload, timeout = 300)
                response.raise_for_status()  # levanta erro se status >= 400
                if self.default_time_sleep > 0:
                    time.sleep(self.default_time_sleep)
                # print("Sucesso:", response.status_code)
                break  # sai do loop se a requisição for bem-sucedida

            except (ConnectionError, RemoteDisconnected) as e:
                retry_count += 1
                if retry_count > max_retries:
                    print("Max retries reached. Exiting...")
                    raise
                print("Erro de conexao:", e)
                print(f"Erro de conexão detectado ({retry_count}/{max_retries}): {e}")
                print("Aguardando 60 segundos antes de tentar novamente...", flush=True)
                time.sleep(60)

            except Exception as e:
                print("Erro inesperado:", e)
                raise
                # break  # interrompe se for outro erro que não de conexão


        # response = requests.post(url, headers=headers, json=payload, timeout = 300)
        # response.raise_for_status()  # dispara exceção se status >= 400

        return response.json()['choices'][0]['message']['content']

    def _query_ollama(self, model: str, model_params: Dict[str, Any], user_prompt: str) -> str:
        if self.host_ollama == 'local':
            client = ollama
        else:
            client = ollama.Client(host=self.host_ollama)


        if self.seed is not None and not 'seed' in model_params:
            model_params['seed'] = self.seed

        response = client.generate(
            model=model,
            prompt=user_prompt,
            stream=False,
            options=model_params,
        )

        return response['response']
    def _save_cache( self ):
        with open(self.cache_path, "w") as f:
            json.dump(self.cache, f, indent=2)

    def query_model(self, model: str, model_params: Dict[str, Any], user_prompt: str, model_type:Literal['ollama', 'openrouter'] = None, max_retries:int=5 ) -> str:
        key = hashlib.md5(user_prompt.encode()+model.encode()+str(model_params).encode()).hexdigest()
        if key in self.cache:
            print( 'USed from cache.', flush=True)
            return self.cache[key]
        try:
            # response_TXT = call_ollama(MODEL, prompt)


            if model_type is None:
                model_type = self.default_model_type
            if model_type == 'ollama':
                response_TXT =  self._query_ollama(model, model_params, user_prompt)
            elif model_type == 'openrouter':
                continua = True
                while continua:
                    try:
                        response_TXT  = self._query_model_raw_openrouter(model, model_params, user_prompt, max_retries=max_retries)
                        continua = False
                    except requests.exceptions.HTTPError as e:
                        if e.response is not None and e.response.status_code == 429:
                            self._save_cache()
                            print("Erro 429: too many requests / rate limit exceeded. Sleeping...")
                            time.sleep( 60 )
                        else:
                            raise
            else:
                raise ValueError(f"Unsupported model type: {model_type}. Use 'ollama' or 'openrouter'.")
        except Exception as e:
            self._save_cache()
            raise

        reply = response_TXT.strip()
        self.cache[key] = reply
        self.cache_counter+=1
        if self.cache_counter % SAVE_CACHE_INTERVAL == 0:
            self._save_cache()
            print('Cache saved to disk.', flush=True)
        # print(f"Cache hit: {self.cache_counter} - {key}", flush=True)
        return reply

    def generate_outputs(self, dataset: List[Dict[str, Any]]) -> Dict[str, List[Dict[str, Any]]]:
        """
        Generate outputs for each model given a labeled dataset.

        Args:
            dataset (List[Dict[str, Any]]): List of data points with 'text' and 'label'.

        Returns:
            Dict[str, List[Dict[str, Any]]]: Outputs organized by model name.
        """
        all_outputs = {model: [] for model in self.models}

        for data_point in tqdm(dataset):
            user_prompt = f"Text: {data_point['text']}\nLabel: {data_point['label']}"

            for model, model_params in zip(self.models, self.models_params):
                try:
                    print(f"Querying model {model}...")
                    output = self.query_model(model, model_params, user_prompt)
                    all_outputs[model].append({
                        "input": data_point,
                        "output": output
                    })
                    time.sleep(1)  # Sleep between requests to be polite
                except Exception as e:
                    print(f"Error querying model {model}: {e}")
                    all_outputs[model].append({
                        "input": data_point,
                        "output": {"error": str(e)}
                    })

        return all_outputs

In [ ]:
DEFAULT_MODEL_PARAMS = {'top_k':1,'temperature':0.1, 'top_p':0.0001}

CACHE_PATH = "/content/drive/MyDrive/codigo_python/cache_llm_respostas_ANALISES.json"
#CACHE_PATH = "/content/drive/MyDrive/codigo_python/cache_llm_respostas_ANALISES.json"
# HOST_OLLAMA = "10.10.10.106:11434"
MY_KEY = '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'

my_llm = LLMInference(
            api_key = MY_KEY,
            use_cache = True,
            cache_interval = 10,
            cache_path = CACHE_PATH,
            default_model_type = 'openrouter', # 'ollama' ou 'openrouter'
            fix_seed = 42,
            host_ollama = 'local',
            default_time_sleep = 1
        )

cache lido


In [ ]:
df=pd.read_csv('/content/math_ODEsdataset_csv.txt')
df.head(5)

,Index,Question,Proposed Solution,Answer
0,0,A population of protozoa grows at a constant r...,"We have dP/dt = kP Therefore dP/dt = 0,7944P ...",235.0000
1,1,A tank contains 1000 L of salt water with 15 k...,Let S(t) be the amount of salt (in kilograms...,12.3000
2,2,A resistor R=10Ω and an inductor L=2H are conn...,"Using Kirchhoff's Voltage Law KVL, the sum of ...",0.0000
3,3,A 5μF capacitor is charged through a 10kΩ resi...,"In an RC charging circuit, Kirchhoff's voltag...",0.0549
4,4,An object is removed from an oven at 200°C and...,Let T(t) represent the temperature of the obje...,114.1900


In [ ]:
def ODE_modelling(model, text):
    prompt = f"""[Instruction]
Assume you are Dr. LLM – an expert in Ordinary Differential Equations (ODEs), with extensive experience in formulating ODE models from natural language descriptions that can be  solved using a solver. Your primary task is to convert a given open-ended math problem into a complete Python code ready to executed. The Python code Should provide a numeric answer when executed.Choose the most appropriate solver (`solve_ivp` or `odeint`) for effectively tackling each Ordinary Differential Equation (ODE) problem.
This Python code should adhere to the following criteria:
1. Your response should be a Python Code that uses one of the following solvers appropriately: `solve_ivp` or  `odeint` based on the nature of the problem.
2. The code should be a final-state approach without displaying intermediate steps or variables.
3. The final result must be rounded to exactly 5 significant figures or 5 decimal places.
4. The response must consist solely of valid Python code, ready to execute without modification.
5. Ensure all LaTeX math expressions in the problem are interpreted and translated into correct mathematical expressions in Python.



Here comes 3  examples:

=====================================================================================================================================================================================================================================================================================================================================================================
[Example 1. A population of protozoa grows at a constant relative rate of 0.44 per individual per day.
On day zero, the population consists of five  protozoa.
Calculate the population size after 8 days?]

Here are the instruction and Python code that solves the given problem and meets the specified requirements:

<think>  **Problem Analysis**:
# Step 1: Interpret the problem.
# - The population grows at a **constant relative rate** of 0.44 per day, which implies exponential growth.
# - This gives the ODE: dP/dt = 0.44 * P, where P(t) is the population at time t (in days).
# - The initial condition is P(0) = 5 (five protozoa at day zero).
# - We need to compute P(8), the population at t = 8 days.

# Step 2: Use solve_ivp to numerically solve the ODE.
# - Define the function f(t, P) = 0.44 * P.
# - Solve over the interval [0, 8] with initial value P(0) = 5.
# - Extract and print the value at t = 8 rounded to 4 decimal places.

</think> (Output-Solution)
from scipy.integrate import solve_ivp
import numpy as np

def model(t, P):
    return 0.44 * P
P0 = [5]
t_span = (0, 8)
sol = solve_ivp(model, t_span, P0, t_eval=[8])
print(round(sol.y[0][0], 4))

=============================================================================================================================================================================================================
[Example 2. A body whose temperature is 20° is placed in a medium which is kept at a constant temperature of 60°. In 5 minute, the body's temperature has risen to 30°. How long will it take for the body's temperature to increase to 40°?]

Here are the instruction and Python code that solves the given problem and meets the specified requirements:

 <think>  **Problem Analysis**:
# Step 1: Newton's Law of Cooling/Heating: dT/dt = -k(T - T_env)
# Here T_env = 60°C (constant medium temperature)
# Initial temperature T(0) = 20°C
# After 5 min, T(5) = 30°C
# Find time t when T(t) = 40°C
# Step 2:
# - Calculate k from the data at t=5
# - Define a residual function for temperature difference at time t
# - Use root finding to get t when temperature reaches 40°C


</think> (Output-Solution)
import numpy as np
from scipy.integrate import odeint
from scipy.optimize import root_scalar

def cooling(T, t, k, T_env):
    return -k * (T - T_env)

def residual(t):
    T_env = 60
    T0 = 20
    k = -np.log((30 - T_env) / (T0 - T_env)) / 5  # calculate k from first data point
    time = np.linspace(0, t, 100)
    sol = odeint(cooling, T0, time, args=(k, T_env))
    T_t = sol[-1][0]
    return T_t - 40

sol = root_scalar(residual, bracket=(0, 50), method='bisect')

print(round(sol.root, 5))

=======================================================================================================================================================================================
[Example 3. A 6μF capacitor is charged through a 30kΩ resistor from a 48V battery. Calculate how long will
it take for the voltage across the capacitor to reach 12V ?]

Here are the instruction and Python code that solves the given problem and meets the specified requirements:

 <think>  **Problem Analysis**:
# Problem: A 6μF capacitor is charged through a 30kΩ resistor from a 48V battery.
# Goal: Find time t when the voltage across the capacitor reaches 12V.
# Physical model: RC charging circuit.
# ODE: dV/dt = (1/RC) * (V_batt - V)
# Initial condition: V(0) = 0
# Target condition: V(t) = 12
# Use solve_ivp to solve numerically and root_scalar to find exact time.

</think>(Output-Solution)

from scipy.integrate import solve_ivp
from scipy.optimize import root_scalar

R = 30000          # Resistance in ohms
C = 6e-6           # Capacitance in farads
V_batt = 48        # Battery voltage in volts
V_target = 12      # Target voltage in volts

def dVdt(t, V):
    return (1 / (R * C)) * (V_batt - V)

sol = solve_ivp(dVdt, t_span=(0, 1), y0=[0], dense_output=True, rtol=1e-9, atol=1e-12)

def voltage_difference(t):
    return sol.sol(t)[0] - V_target

result = root_scalar(voltage_difference, bracket=[0, 1], method='brentq')

print(f"(result.root:.5f)")



========================================================================================================================================================================================================
The following sentences  are forbidden:
”
This code solves the  problem for the given differential equation using 'solve_ivp' from SciPy;
This code solves the  problem for the given differential equation using ‘odeint‘ from SciPy;

Please avoid the above sentences.
========================================================================================================================================================================================================
Take it easy and have a deep breath before answering the question. This won't be a challenge for you atall.


Here comes the question:
{text}

Your response:
"""
    response = my_llm.query_model( model, DEFAULT_MODEL_PARAMS, prompt)
    print("**************", model)
    print(response)
    print("************")
    return response

In [ ]:
text = df.iloc[0]['Question']
model = 'mistralai/mistral-small-3.2-24b-instruct:free'
ODE_modelling(model, text)

USed from cache.
************** mistralai/mistral-small-3.2-24b-instruct:free
from scipy.integrate import solve_ivp

def model(t, P):
    return 0.7944 * P

P0 = [2]
t_span = (0, 6)
sol = solve_ivp(model, t_span, P0, t_eval=[6])
print(round(sol.y[0][0], 5))
************


'from scipy.integrate import solve_ivp\n\ndef model(t, P):\n    return 0.7944 * P\n\nP0 = [2]\nt_span = (0, 6)\nsol = solve_ivp(model, t_span, P0, t_eval=[6])\nprint(round(sol.y[0][0], 5))'

In [ ]:
import pandas as pd
import os
import hashlib
import json
from pathlib import Path

# Caminho do arquivo de cache
CACHE_FILE = Path('/content/drive/MyDrive/codigo_python/math_ODEsdataset_csv.txt')

# Garante que o diretório de cache existe
CACHE_FILE.parent.mkdir(parents=True, exist_ok=True)

# Tenta carregar o cache existente
cache = {}
if CACHE_FILE.exists():
    print('🔁 Cache lido dos códigos!')
    try:
        with open(CACHE_FILE, 'r') as f:
            cache = json.load(f)
        print(f"🔹 Cache carregado com {len(cache)} entradas")
    except Exception as e:
        print(f"⚠️ Erro ao carregar cache: {e}")
        print("🔸 Iniciando cache novo.")

def get_cache_key(model, question):
    """Gera chave única para cache"""
    return hashlib.md5(f"{model}_{question}".encode()).hexdigest()

# Carrega o dataset
df = pd.read_csv('/content/math_ODEsdataset_csv.txt')

# Lista de modelos
models = [
    'meta-llama/llama-3.1-405b-instruct:free',
    'google/gemma-3-27b-it:free',
    'deepseek/deepseek-r1-distill-llama-70b:free',
    'mistralai/mistral-small-3.2-24b-instruct:free',
    'x-ai/grok-3-mini-beta',
    'google/gemini-2.5-pro-preview'
]

# Processa cada modelo
for model in models:
    responses = []
    print(f"\n{'='*60}")
    print(f"▶️ Processando modelo: {model}")
    print(f"{'='*60}")

    model_col = model + '_class'
    existing_responses = df[model_col].tolist() if model_col in df.columns else [None] * len(df)

    for index, row in df.iterrows():
        question = str(row['Question'])
        cache_key = get_cache_key(model, question)

        # Usa cache
        if cache_key in cache:
            responses.append(cache[cache_key])
            print(f"✓ Cache usado para índice {index}")
            continue

        # Usa resposta existente (não é 'error' nem vazia)
        if existing_responses[index] not in [None, 'error', '']:
            responses.append(existing_responses[index])
            cache[cache_key] = existing_responses[index]
            print(f"↻ Reuso da resposta existente para índice {index}")
            continue

        # Gera nova resposta
        result = 'error'
        for attempt in range(3):
            try:
                rsp = ODE_modelling(model, question)
                if rsp != 'error':
                    result = rsp
                    break
            except Exception as e:
                print(f"⚠️ Erro na tentativa {attempt+1} no índice {index}: {e}")

        responses.append(result)
        cache[cache_key] = result

        # Salva o cache a cada 20 linhas
        if (index + 1) % 20 == 0:
            with open(CACHE_FILE, 'w') as f:
                json.dump(cache, f)
            print(f"💾 Cache salvo após linha {index+1}")

    # Atualiza coluna do modelo
    df[model_col] = responses

    # Salva cache após cada modelo
    with open(CACHE_FILE, 'w') as f:
        json.dump(cache, f)
    print(f"✅ Cache salvo após processar modelo: {model}")

# Garante que a pasta 'results' existe
os.makedirs('results', exist_ok=True)

# Salva o CSV final
df.to_csv('math_ODEsdataset_csv.txt_models', index=False)
print("🎯 Processamento completo! CSV salvo.")


🔁 Cache lido dos códigos!
🔹 Cache carregado com 608 entradas

▶️ Processando modelo: meta-llama/llama-3.1-405b-instruct:free
✓ Cache usado para índice 0
✓ Cache usado para índice 1
✓ Cache usado para índice 2
✓ Cache usado para índice 3
✓ Cache usado para índice 4
✓ Cache usado para índice 5
✓ Cache usado para índice 6
✓ Cache usado para índice 7
✓ Cache usado para índice 8
✓ Cache usado para índice 9
✓ Cache usado para índice 10
✓ Cache usado para índice 11
✓ Cache usado para índice 12
✓ Cache usado para índice 13
✓ Cache usado para índice 14
✓ Cache usado para índice 15
✓ Cache usado para índice 16
✓ Cache usado para índice 17
✓ Cache usado para índice 18
✓ Cache usado para índice 19
✓ Cache usado para índice 20
✓ Cache usado para índice 21
✓ Cache usado para índice 22
✓ Cache usado para índice 23
✓ Cache usado para índice 24
✓ Cache usado para índice 25
✓ Cache usado para índice 26
✓ Cache usado para índice 27
✓ Cache usado para índice 28
✓ Cache usado para índice 29
✓ Cache usado p

==================================================================

In [ ]:
df.iloc['']

In [ ]:
import pandas as pd
import os
import re
import sympy as sp
import numpy as np
from scipy.integrate import solve_ivp, odeint
import sys
import contextlib
from io import StringIO

# === Configurações iniciais ===
#INPUT_CSV = '/content/drive/MyDrive/codigo_python/math_ODEsdataset_results.csv'
OUTPUT_CSV = '/content/drive/MyDrive/codigo_python/math_ODEsdataset_final_results.csv'

# === Ambiente simplificado para execução ===
# Mantemos apenas as funções essenciais para resolver EDOs
DEFAULT_GLOBALS = {
    'np': np,
    'sp': sp,
    'solve_ivp': solve_ivp,
    'odeint': odeint,
    'dsolve': sp.dsolve,
    'Eq': sp.Eq,
    'Function': sp.Function,
    'Symbol': sp.Symbol,
    'sin': sp.sin,
    'cos': sp.cos,
    'exp': sp.exp,
    'log': sp.log,
    'sqrt': sp.sqrt,
    'pi': sp.pi,
    'e': sp.E
}

# === Funções auxiliares ===
def extract_code(text):
    """Remove formatação Markdown do código"""
    if not isinstance(text, str):
        return ''
    return re.sub(r'```(python)?|```', '', text).strip()

@contextlib.contextmanager
def capture_stdout():
    """Captura saída impressa durante execução"""
    buffer = StringIO()
    original_stdout = sys.stdout
    sys.stdout = buffer
    try:
        yield buffer
    finally:
        sys.stdout = original_stdout

def execute_ode_code(code_str):
    """Executa código Python para soluções de EDOs"""
    try:
        local_vars = {}

        # Execução direta sem restrições excessivas
        with capture_stdout() as stdout:
            exec(code_str, DEFAULT_GLOBALS, local_vars)
        output = stdout.getvalue().strip()

        # Tenta extrair número da saída
        if output:
            try:
                return float(output)
            except ValueError:
                pass

        # Tenta encontrar resultado em variáveis de saída
        output_vars = ['result', 'solution', 'value', 'final_value', 'answer', 'output']
        for name in output_vars:
            if name in local_vars and isinstance(local_vars[name], (int, float)):
                return float(local_vars[name])

        # Tenta encontrar o último valor numérico
        for val in local_vars.values():
            if isinstance(val, (int, float)):
                return float(val)

        return None
    except Exception as e:
        print(f"Erro na execução: {str(e)[:100]}")
        return None

# === Carregar e processar o dataset ===
def process_dataset():
    # Carregar CSV
    try:
        df = pd.read_csv('/content/math_ODEsdataset_csv.txt_models')
        print(f"Dataset carregado: {len(df)} linhas")
    except FileNotFoundError:
        print(f"Erro: Arquivo não encontrado - {INPUT_CSV}")
        return

    # Identificar colunas com códigos
    code_columns = [col for col in df.columns if col.endswith('_class')]

    if not code_columns:
        print("Nenhuma coluna de código encontrada (_class)")
        return

    print(f"Colunas de código identificadas: {', '.join(code_columns)}")

    # Processar cada coluna de código
    for col in code_columns:
        result_col = col.replace('_class', '_exec_result')
        df[result_col] = 'not_executed'

        print(f"\n{'='*50}")
        print(f"Processando coluna: {col}")
        print(f"{'='*50}")

        for idx, row in df.iterrows():
            code_raw = row[col]

            if pd.isna(code_raw):
                df.at[idx, result_col] = 'no_code'
                continue


            code_clean = extract_code(str(code_raw))

            print(f'Code executing ({col}/{idx}):', code_clean, flush=True)

            if not code_clean:
                df.at[idx, result_col] = 'empty_code'
                continue

            # Executar código
            result = execute_ode_code(code_clean)

            if result is not None:
                df.at[idx, result_col] = round(result, 4)
            else:
                df.at[idx, result_col] = 'exec_error'

            # Mostrar progresso
            if (idx + 1) % 10 == 0:
                print(f"Processadas {idx+1}/{len(df)} linhas")

    # Salvar resultados
    os.makedirs('results', exist_ok=True)
    df.to_csv(OUTPUT_CSV, index=False)
    print(f"\n✅ Resultados salvos em: {OUTPUT_CSV}")

    return df

# === Execução principal ===
if __name__ == "__main__":
    print("Iniciando execução de códigos Python do dataset...")
    result_df = process_dataset()

    if result_df is not None:
        # Mostrar resumo estatístico
        print("\nResumo estatístico:")
        for col in result_df.columns:
            if col.endswith('_exec_result'):
                success_count = result_df[col].apply(lambda x: isinstance(x, float)).sum()
                error_count = len(result_df) - success_count
                print(f"{col}: {success_count} sucessos, {error_count} erros")

    print("Processamento concluído!")

Iniciando execução de códigos Python do dataset...
Dataset carregado: 50 linhas
Colunas de código identificadas: meta-llama/llama-3.1-405b-instruct:free_class, google/gemma-3-27b-it:free_class, deepseek/deepseek-r1-distill-llama-70b:free_class, mistralai/mistral-small-3.2-24b-instruct:free_class, x-ai/grok-3-mini-beta_class, google/gemini-2.5-pro-preview_class

Processando coluna: meta-llama/llama-3.1-405b-instruct:free_class
Code executing (meta-llama/llama-3.1-405b-instruct:free_class/0): from scipy.integrate import solve_ivp
import numpy as np

def model(t, P):
    return 0.7944 * P

P0 = [2]
t_span = (0, 6)
sol = solve_ivp(model, t_span, P0, t_eval=[6])
print(round(sol.y[0][0], 5))
Code executing (meta-llama/llama-3.1-405b-instruct:free_class/1): from scipy.integrate import solve_ivp
import numpy as np

def model(t, S):
    return - (10/1000) * S

S0 = [15]
t_span = (0, 20)
sol = solve_ivp(model, t_span, S0, t_eval=[20])
print(round(sol.y[0][0], 5))
Code executing (meta-llama/llama

<string>:19: ODEintWarning: Excess work done on this call (perhaps wrong Dfun type). Run with full_output = 1 to get quantitative information.


A saída de streaming foi truncada nas últimas 5000 linhas.
# Solve the ODE
sol = odeint(cooling, T0, t, args=(k, T_env))

# Extract the temperature at t=20 minutes
T_20 = sol[-1][0]

print(round(T_20, 5))
Code executing (google/gemma-3-27b-it:free_class/24): import numpy as np
from scipy.integrate import odeint
from scipy.optimize import root_scalar

def decay(A, t, k):
    return -k * A

def residual(t):
    A0 = 100
    k = -np.log(60 / A0) / 10
    A_t = A0 * np.exp(-k * t)
    return A_t - 50

sol = root_scalar(residual, bracket=(0, 20), method='bisect')

print(round(sol.root, 5))
Code executing (google/gemma-3-27b-it:free_class/25): from scipy.integrate import solve_ivp
import numpy as np

L = 0.01  # Inductance in Henries
R = 20  # Resistance in Ohms
V = 100  # Voltage in Volts

def dIdt(t, I):
    return (V - R * I) / L

I0 = [0]  # Initial current is 0
t_span = (0, 0.01)
sol = solve_ivp(dIdt, t_span, I0, dense_output=True)

print(round(sol.y[0][0], 5))
Erro na execução: name 'V

In [ ]:
df=pd.read_csv('/content/drive/MyDrive/codigo_python/math_ODEsdataset_final_results.csv')

In [ ]:
df

,Index,Question,Proposed Solution,Answer,meta-llama/llama-3.1-405b-instruct:free_class,google/gemma-3-27b-it:free_class,deepseek/deepseek-r1-distill-llama-70b:free_class,mistralai/mistral-small-3.2-24b-instruct:free_class,x-ai/grok-3-mini-beta_class,google/gemini-2.5-pro-preview_class,meta-llama/llama-3.1-405b-instruct:free_exec_result,google/gemma-3-27b-it:free_exec_result,deepseek/deepseek-r1-distill-llama-70b:free_exec_result,mistralai/mistral-small-3.2-24b-instruct:free_exec_result,x-ai/grok-3-mini-beta_exec_result,google/gemini-2.5-pro-preview_exec_result
0,0,A population of protozoa grows at a constant r...,"We have dP/dt = kP Therefore dP/dt = 0,7944P ...",2.350000e+02,```python\nfrom scipy.integrate import solve_i...,".y[0][0], 4))\n===============================...",from scipy.integrate import solve_ivp\nimport ...,from scipy.integrate import solve_ivp\n\ndef m...,from scipy.integrate import solve_ivp\n\ndef m...,```python\nfrom scipy.integrate import solve_i...,234.9685,exec_error,234.9685,234.9685,234.9685,234.9685
1,1,A tank contains 1000 L of salt water with 15 k...,Let S(t) be the amount of salt (in kilograms...,1.230000e+01,```python\nfrom scipy.integrate import solve_i...,```python\nfrom scipy.integrate import solve_i...,from scipy.integrate import odeint\nimport num...,from scipy.integrate import solve_ivp\n\ndef s...,```python\nfrom scipy.integrate import solve_i...,```python\nimport numpy as np\nfrom scipy.inte...,12.281,12.281,15.0,12.281,12.281,12.281
2,2,A resistor R=10Ω and an inductor L=2H are conn...,"Using Kirchhoff's Voltage Law KVL, the sum of ...",0.000000e+00,```python\nimport numpy as np\nfrom scipy.inte...,is V(t) = V_batt * (1 - e^(-t/(RC)))\n# Step 2...,To determine the initial current \( I(0) \) wh...,from scipy.integrate import solve_ivp\n\nR = 1...,from scipy.integrate import solve_ivp\n\ndef m...,```python\nimport numpy as np\nfrom scipy.inte...,exec_error,exec_error,exec_error,exec_error,0.0,exec_error
3,3,A 5μF capacitor is charged through a 10kΩ resi...,"In an RC charging circuit, Kirchhoff's voltag...",5.490000e-02,```python\nfrom scipy.integrate import solve_i...,```python\nfrom scipy.integrate import solve_i...,from scipy.integrate import solve_ivp\nfrom sc...,```python\nfrom scipy.integrate import solve_i...,from scipy.integrate import solve_ivp\nfrom sc...,```python\nfrom scipy.integrate import solve_i...,exec_error,exec_error,exec_error,exec_error,exec_error,exec_error
4,4,An object is removed from an oven at 200°C and...,Let T(t) represent the temperature of the obje...,1.141900e+02,```python\nimport numpy as np\nfrom scipy.inte...,```python\nfrom scipy.integrate import solve_i...,from scipy.integrate import odeint\nimport num...,from scipy.integrate import odeint\nimport num...,```python\nimport numpy as np\nfrom scipy.inte...,```python\nimport numpy as np\nfrom scipy.inte...,114.2857,210.0942,114.2857,114.2857,exec_error,114.2886
5,5,A certain village in Angola had a population ...,Let P(t) be the population at time t. The rate...,9.260000e+00,```python\nimport numpy as np\nfrom scipy.inte...,```python\nfrom scipy.integrate import solve_i...,from scipy.integrate import solve_ivp\nfrom sc...,from scipy.integrate import odeint\nfrom scipy...,from scipy.integrate import solve_ivp\nfrom sc...,```python\nimport numpy as np\nfrom scipy.inte...,exec_error,exec_error,exec_error,exec_error,exec_error,exec_error
6,6,"Find the charge when t=0,2, in a series RC cir...","We have Rdq(t)/dt+1/C q=V Since R=10, C=4*10^(...",-5.290000e-02,```python\nimport numpy as np\nfrom scipy.inte...,```python\nfrom scipy.integrate import solve_i...,from scipy.integrate import solve_ivp\nimport ...,Here's the Python code that solves the problem...,from scipy.integrate import solve_ivp\nimport ...,```python\nimport numpy as np\nfrom scipy.inte...,10.0,exec_error,-132.7955,exec_error,exec_error,-0.0534
7,7,A resistor R=12Ω and an inductor L=4H are conn...,"Using Kirchhoff's Voltage Law (KVL), the sum o...",2

In [ ]:
df.drop([

    'Proposed Solution',
    'mistralai/mistral-small-3.2-24b-instruct:free_class',
    'deepseek/deepseek-r1-distill-llama-70b:free_class',
    'google/gemma-3-27b-it:free_class',
    'meta-llama/llama-3.1-405b-instruct:free_class',
    'x-ai/grok-3-mini-beta_class',
    'google/gemini-2.5-pro-preview_class'
], axis=1, inplace=True)


In [ ]:
df

,Index,Question,Answer,meta-llama/llama-3.1-405b-instruct:free_exec_result,google/gemma-3-27b-it:free_exec_result,deepseek/deepseek-r1-distill-llama-70b:free_exec_result,mistralai/mistral-small-3.2-24b-instruct:free_exec_result,x-ai/grok-3-mini-beta_exec_result,google/gemini-2.5-pro-preview_exec_result
0,0,A population of protozoa grows at a constant r...,2.350000e+02,234.9685,exec_error,234.9685,234.9685,234.9685,234.9685
1,1,A tank contains 1000 L of salt water with 15 k...,1.230000e+01,12.281,12.281,15.0,12.281,12.281,12.281
2,2,A resistor R=10Ω and an inductor L=2H are conn...,0.000000e+00,exec_error,exec_error,exec_error,exec_error,0.0,exec_error
3,3,A 5μF capacitor is charged through a 10kΩ resi...,5.490000e-02,exec_error,exec_error,exec_error,exec_error,exec_error,exec_error
4,4,An object is removed from an oven at 200°C and...,1.141900e+02,114.2857,210.0942,114.2857,114.2857,exec_error,114.2886
5,5,A certain village in Angola had a population ...,9.260000e+00,exec_error,exec_error,exec_error,exec_error,exec_error,exec_error
6,6,"Find the charge when t=0,2, in a series RC cir...",-5.290000e-02,10.0,exec_error,-132.7955,exec_error,exec_error,-0.0534
7,7,A resistor R=12Ω and an inductor L=4H are conn...,2.660000e-01,exec_error,0.2662,exec_error,0.2662,0.2662,0.2662
8,8,A roast chicken is removed from the oven when ...,1.364000e+02,exec_error,exec_error,exec_error,exec_error,exec_error,exec_error
9,9,"Escherichia coli, a common bacteria found in t...",1.676300e+02,exec_error,exec_error,exec_error,exec_error,exec_error,exec_error


In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import sklearn

print(f"🔍 Versão do scikit-learn: {sklearn.__version__}")

ABS_TOL = 0.1
REL_TOL = 0.05

df = df  # Substitua pelo caminho correto
answer_col = "Answer"
model_cols = [col for col in df.columns if col.endswith('_exec_result')]

metrics = []

for model in model_cols:
    y_true = []
    y_pred = []
    acc_count = 0
    total = 0

    for truth, pred in zip(df[answer_col], df[model]):
        total += 1
        if isinstance(pred, str) and "exec_error" in pred:
            continue
        try:
            pred = float(pred)
            truth = float(truth)
        except:
            continue

        y_true.append(truth)
        y_pred.append(pred)

        if abs(pred - truth) <= ABS_TOL or abs(pred - truth) / max(abs(truth), 1e-8) <= REL_TOL:
            acc_count += 1

    if len(y_true) == 0:
        metrics.append({
            "model": model,
            "accuracy (%)": 0.0 if total > 0 else None,
            "mae (%)": None,
            "mse (%)": None,
            "rmse (%)": None,
            "r2 (%)": None,


        })
        continue

    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)

    metrics.append({
        "model": model,
        "accuracy (%)": round((acc_count / total) * 100,1),
        "mae (%)": round(mae * 100,1),
        "mse (%)": round(mse * 100,1),
        "rmse (%)": round(rmse * 100,1),
        "r2 (%)": round(r2 * 100,1 ),


    })

metrics_df = pd.DataFrame(metrics)
metrics_df = metrics_df.sort_values(by="accuracy (%)", ascending=False)

print("📊 Métricas de avaliação por modelo (em %):")
print(metrics_df.to_string(index=False))


🔍 Versão do scikit-learn: 1.6.1
📊 Métricas de avaliação por modelo (em %):
                                                    model  accuracy (%)  mae (%)      mse (%)  rmse (%)  r2 (%)
                google/gemini-2.5-pro-preview_exec_result          54.0 348198.1 2.142477e+10 1463720.2   100.0
                   google/gemma-3-27b-it:free_exec_result          30.0 442224.4 2.719301e+10 1649030.2   100.0
mistralai/mistral-small-3.2-24b-instruct:free_exec_result          28.0 309137.1 2.252562e+10 1500853.7   100.0
                        x-ai/grok-3-mini-beta_exec_result          26.0 435354.3 3.180083e+10 1783278.6   100.0
      meta-llama/llama-3.1-405b-instruct:free_exec_result          24.0 537684.0 3.258128e+10 1805028.4   100.0
  deepseek/deepseek-r1-distill-llama-70b:free_exec_result          24.0 230804.2 9.246117e+09  961567.3   100.0
